# Example 05B: Run Ensemble Simulations

This notebook resumes the prepared siblings from notebook 05A and runs the expensive dynamic portion: step 10 input definition, step 11 SWMM simulation, and step 12 flow-component extraction.

## Setup

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris as sp

BASE_PROJECT_FILE = EXAMPLES_DIR / "output_example_2_project" / "sewertris_project.json"
ENSEMBLE_ROOT = EXAMPLES_DIR / "output_example_5_ensembles"
SCENARIO_NAME = "bwf_gwi_rdii"

base_project = sp.SewerTrisProject.load(BASE_PROJECT_FILE)
ENSEMBLE_ROOT.mkdir(parents=True, exist_ok=True)
print("Base project:", base_project.project_file)

## Load Prepared Siblings

In [ ]:
RUN_SIMULATIONS = True
USE_PARALLEL = True
MAX_WORKERS = 6

PREPARED_MANIFEST = ENSEMBLE_ROOT / "prepared_siblings_manifest.csv"
SIMULATION_MANIFEST = ENSEMBLE_ROOT / "simulation_manifest.csv"

prepared_manifest = pd.read_csv(PREPARED_MANIFEST)
prepared_specs = prepared_manifest[["ensemble", "realization", "project_file"]].to_dict("records")
for spec in prepared_specs:
    stem = f"simulate_{spec['ensemble']}_{int(spec['realization']):03d}"
    spec["progress_path"] = ENSEMBLE_ROOT / "_workers" / f"{stem}_progress.json"
display(prepared_manifest)

## Parallel Simulation

Each worker loads one prepared sibling and resumes at step 10. Keeping this separate makes failed or slow SWMM runs easier to restart without rebuilding the physical model.

In [ ]:
def json_ready(value):
    if isinstance(value, Path):
        return str(value.resolve())
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(v) for v in value]
    return value


def run_worker_payload(payload, stem):
    worker_dir = ENSEMBLE_ROOT / "_workers"
    worker_dir.mkdir(parents=True, exist_ok=True)
    payload_path = worker_dir / f"{stem}.json"
    result_path = worker_dir / f"{stem}_result.json"
    payload_path.write_text(json.dumps(json_ready(payload), indent=2))

    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(SRC_DIR), env.get("PYTHONPATH", "")]).strip(os.pathsep)
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "sewertris.ensemble",
            str(payload_path),
            "--result-file",
            str(result_path),
        ],
        cwd=PROJECT_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"Worker failed for {stem}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}")
    return json.loads(result_path.read_text())


def read_progress(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None


def progress_label(spec):
    return f"{spec['ensemble']} run_{int(spec['realization']):03d}"


def print_active_progress(pending_specs, completed, successful, failed):
    total = completed + len(pending_specs)
    active = []
    for spec in pending_specs:
        event = read_progress(spec["progress_path"])
        if event is None:
            active.append(f"{progress_label(spec)}: queued")
        else:
            step = event.get("step", "starting")
            active.append(f"{progress_label(spec)}: {step}")
    preview = "; ".join(active[:6])
    if len(active) > 6:
        preview += f"; ... +{len(active) - 6} more"
    print(
        f"[progress] completed {completed}/{total} "
        f"(success={successful}, failed={failed}). Active: {preview}"
    )


def run_parallel(items, worker, max_workers, heartbeat_seconds=10):
    results = []
    successful = 0
    failed = 0
    last_heartbeat = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(worker, item): item for item in items}
        pending = set(futures)

        while pending:
            finished = [future for future in pending if future.done()]
            if finished:
                for future in finished:
                    pending.remove(future)
                    spec = futures[future]
                    try:
                        result = future.result()
                    except Exception:
                        failed += 1
                        print(f"[failed] {progress_label(spec)}")
                        raise
                    else:
                        successful += 1
                        results.append(result)
                        event = read_progress(spec["progress_path"]) or {}
                        print(
                            f"[done] {progress_label(spec)} "
                            f"({successful}/{len(items)} successful) "
                            f"last_step={event.get('step', result.get('rerun_from'))}"
                        )
                continue

            now = time.time()
            if now - last_heartbeat >= heartbeat_seconds:
                pending_specs = [futures[future] for future in pending]
                print_active_progress(
                    pending_specs,
                    completed=len(items) - len(pending),
                    successful=successful,
                    failed=failed,
                )
                last_heartbeat = now
            time.sleep(1)

    return results


def run_simulation(spec):
    stem = f"simulate_{spec['ensemble']}_{int(spec['realization']):03d}"
    payload = {
        "mode": "simulation",
        "parent_project_file": BASE_PROJECT_FILE,
        "scenario_name": SCENARIO_NAME,
        "progress_path": spec["progress_path"],
        "spec": {key: value for key, value in spec.items() if key != "progress_path"},
    }
    return run_worker_payload(payload, stem)


if RUN_SIMULATIONS:
    if USE_PARALLEL:
        simulation_results = run_parallel(prepared_specs, run_simulation, MAX_WORKERS)
    else:
        simulation_results = [run_simulation(spec) for spec in prepared_specs]

    simulation_manifest = pd.DataFrame(simulation_results).sort_values(["ensemble", "realization"])
    simulation_manifest.to_csv(SIMULATION_MANIFEST, index=False)
    display(simulation_manifest)
else:
    print("Set RUN_SIMULATIONS = True to run the SWMM simulations.")

## Analyze Results

In [ ]:
if SIMULATION_MANIFEST.exists():
    simulation_manifest = pd.read_csv(SIMULATION_MANIFEST)
    simulation_manifest["flows_exists"] = simulation_manifest["flows_path"].map(lambda p: Path(p).exists())
    display(simulation_manifest)
else:
    print("No simulation manifest yet.")

## Example Single-Member Comparison

In [ ]:
if SIMULATION_MANIFEST.exists() and not simulation_manifest.empty:
    first = simulation_manifest.iloc[0]
    first_project = sp.SewerTrisProject.load(first["project_file"])
    options = base_project.step_parameters("10_dynamic_flow_input_definition_base_model").get("options_dict", {})
    start = f"{options.get('START_DATE', '01/01/1990')} {options.get('START_TIME', '00:00:00')}"
    end = f"{options.get('END_DATE', '01/10/1990')} {options.get('END_TIME', '00:00:00')}"

    sp.plot_two_models(
        "flow_components",
        base_project,
        first_project,
        labels=("Original Stillwater", first["ensemble"]),
        scenario_name=SCENARIO_NAME,
        start=start,
        end=end,
    )
else:
    print("Run the simulations first, then compare a completed member.")